# Parte 2 — planejamento com Qwen3.5-0.8B

Hipótese: LoRA melhora a proporção de planos que obedecem aos requisitos estruturais do MiniCode. Compare o mesmo modelo antes e depois do ajuste no conjunto de teste fixo. **Ative GPU no Colab.** O conjunto contém 80 exemplos de treino, 10 de validação e 10 de teste. Validação e teste continuam pequenos: reporte resultados como exploratórios. Veja notebooks/data/README.md para origem, limitações e rubrica manual. Não use o teste para escolher configurações de treinamento.


In [ ]:
%pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git@main" accelerate peft bitsandbytes matplotlib


In [ ]:
import json, subprocess, sys, torch, matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

REPO = Path('/content/minicode')
BRANCH = 'feat/parte-2-qwen-planejamento'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/joaovicentedev/minicode.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
if not (REPO / 'notebooks/data/train.jsonl').exists():
    raise FileNotFoundError('Publique a branch antes de executar o notebook.')
sys.path.insert(0, str(REPO / 'src'))
from minicode.planning_data import load_examples, format_example, plan_score, summarize_plans
train = load_examples(REPO / 'notebooks/data/train.jsonl')
validation = load_examples(REPO / 'notebooks/data/validation.jsonl')
test = load_examples(REPO / 'notebooks/data/test.jsonl')
rubrics = json.loads((REPO / 'notebooks/data/rubrics.json').read_text(encoding='utf-8'))
print('Tamanhos:', len(train), len(validation), len(test))
assert (len(train), len(validation), len(test)) == (80, 10, 10)
assert torch.cuda.is_available(), 'Selecione um runtime com GPU.'


In [ ]:
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, quantization_config=quantization, device_map='auto')
model.config.use_cache = False


In [ ]:
def encode(example):
    pair = format_example(example)
    prefix = tokenizer.apply_chat_template([{'role': 'user', 'content': pair['prompt']}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    prompt_ids = tokenizer(prefix, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(pair['completion'] + tokenizer.eos_token, add_special_tokens=False)['input_ids']
    ids = (prompt_ids + answer_ids)[:1536]
    labels = [-100] * min(len(prompt_ids), len(ids)) + ids[len(prompt_ids):]
    return {'input_ids': ids, 'labels': labels}

encoded_train = [encode(item) for item in train]
encoded_validation = [encode(item) for item in validation]
assert all(any(label != -100 for label in row['labels']) for row in encoded_train + encoded_validation)

def batch(rows):
    width = max(len(row['input_ids']) for row in rows)
    ids = [row['input_ids'] + [tokenizer.pad_token_id] * (width - len(row['input_ids'])) for row in rows]
    labels = [row['labels'] + [-100] * (width - len(row['labels'])) for row in rows]
    return {'input_ids': torch.tensor(ids, device=model.device), 'attention_mask': torch.tensor([[int(token != tokenizer.pad_token_id) for token in line] for line in ids], device=model.device), 'labels': torch.tensor(labels, device=model.device)}


## Avaliação local
`structural_pass` mede atendimento conjunto aos requisitos de formato. `task_alignment`, `spec_coverage` e `context_consistency` verificam grupos de termos específicos em `rubrics.json`; `quality_pass` exige as três dimensões e `overall_pass` exige estrutura e evidências. Essas taxas são proxies lexicais: não comprovam correção semântica e podem rejeitar sinônimos válidos ou aceitar contradições.
As rubricas são metadados exclusivos da avaliação: não entram no prompt nem no treino. Após a execução, revise os planos salvos em `predictions.json` com a rubrica manual de `notebooks/data/README.md`, registrando justificativas. Mantenha a mesma rubrica para base e adaptado e não ajuste critérios após observar respostas de teste.


In [ ]:
@torch.no_grad()
def generate_plans(active_model, examples):
    active_model.eval()
    answers = []
    for example in examples:
        prompt = format_example(example)['prompt']
        text = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(text, return_tensors='pt').to(active_model.device)
        output = active_model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        answers.append(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    return answers

# Baseline no mesmo conjunto de teste, antes de ajustar os pesos.
base_answers = generate_plans(model, test)
base_metrics = summarize_plans(base_answers, test, rubrics)
print('Base:', base_metrics)


In [ ]:
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=2e-4)
train_loss, validation_loss = [], []
for epoch in range(3):
    model.train()
    epoch_losses = []
    for row in encoded_train:
        optimizer.zero_grad()
        loss = model(**batch([row])).loss
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    train_loss.append(sum(epoch_losses) / len(epoch_losses))
    model.eval()
    with torch.no_grad():
        losses = [model(**batch([row])).loss.item() for row in encoded_validation]
    validation_loss.append(sum(losses) / len(losses))
    print(epoch + 1, train_loss[-1], validation_loss[-1])


In [ ]:
plt.plot(range(1, len(train_loss)+1), train_loss, label='Treino')
plt.plot(range(1, len(validation_loss)+1), validation_loss, label='Validação')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.legend(); plt.show()
ft_answers = generate_plans(model, test)
ft_metrics = summarize_plans(ft_answers, test, rubrics)
print('Base:', base_metrics)
print('LoRA:', ft_metrics)
for example, before, after in zip(test, base_answers, ft_answers):
    print('\nTarefa:', example['task'], '\nBase:', before, '\nLoRA:', after)


In [ ]:
OUTPUT = Path('/content/minicode-plan-lora')
model.save_pretrained(OUTPUT)
processor.save_pretrained(OUTPUT)
(OUTPUT / 'metrics.json').write_text(json.dumps({'base': base_metrics, 'lora': ft_metrics, 'train_loss': train_loss, 'validation_loss': validation_loss}, indent=2), encoding='utf-8')
print('Adaptador e métricas:', OUTPUT)

# Preserve evidências por tarefa para auditar a métrica e aplicar a rubrica manual.
review = []
for example, before, after in zip(test, base_answers, ft_answers, strict=True):
    review.append({
        'example': example, 'rubric': rubrics[example['task']],
        'base': {'plan': before, 'scores': plan_score(before, rubrics[example['task']])},
        'lora': {'plan': after, 'scores': plan_score(after, rubrics[example['task']])},
        'manual_review': {'base': None, 'lora': None},
    })
(OUTPUT / 'predictions.json').write_text(json.dumps(review, ensure_ascii=False, indent=2), encoding='utf-8')


## Avaliação no agente completo e ablação

Sirva o modelo base e o adaptador em endpoints compatíveis com OpenAI. Para comparar, execute `minicode-eval run` e `grade` nos mesmos casos, uma vez com `MINICODE_PLAN_MODEL` apontando para o modelo base e outra para o adaptador. Execute também o fluxo padrão sem endpoint de planejamento especializado. Registre `TaskSuccessRate` de cada execução e mantenha os relatórios. A ablação usa o fluxo padrão versus o fluxo com modelo de planejamento separado. Nenhum resultado end-to-end é declarado aqui antes de executar esses comandos.
